In [14]:
from moabb.paradigms import FilterBankMotorImagery, FilterBankLeftRightImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

sfreq=250
paradigm=FilterBankMotorImagery(resample=sfreq)
{d.code:d for d in paradigm.datasets}


{'BNCI2014-001': <moabb.datasets.bnci.BNCI2014_001 at 0x1465c0524f10>,
 'BNCI2014-004': <moabb.datasets.bnci.BNCI2014_004 at 0x1465c00bc390>,
 'Cho2017': <moabb.datasets.gigadb.Cho2017 at 0x1465c02783d0>,
 'GrosseWentrup2009': <moabb.datasets.mpi_mi.GrosseWentrup2009 at 0x1465c87a6010>,
 'Lee2019-MI': <moabb.datasets.Lee2019.Lee2019_MI at 0x1465c022ce50>,
 'Liu2024': <moabb.datasets.liu2024.Liu2024 at 0x146627455810>,
 'PhysionetMotorImagery': <moabb.datasets.physionet_mi.PhysionetMI at 0x1465c0279f90>,
 'Schirrmeister2017': <moabb.datasets.schirrmeister2017.Schirrmeister2017 at 0x1465c0280690>,
 'Shin2017A': <moabb.datasets.bbci_eeg_fnirs.Shin2017A at 0x1465c87bf410>,
 'Stieger2021': <moabb.datasets.stieger2021.Stieger2021 at 0x1465c10b0350>,
 'Weibo2014': <moabb.datasets.Weibo2014.Weibo2014 at 0x1465c0281050>,
 'Zhou2016': <moabb.datasets.Zhou2016.Zhou2016 at 0x1465c00bff90>}

Choosing from all possible events


{'AlexandreMotorImagery': <moabb.datasets.alex_mi.AlexMI at 0x1465c085cb90>,
 'BNCI2014-001': <moabb.datasets.bnci.BNCI2014_001 at 0x1465c02a8f90>,
 'BNCI2014-002': <moabb.datasets.bnci.BNCI2014_002 at 0x1465c026bcd0>,
 'BNCI2014-004': <moabb.datasets.bnci.BNCI2014_004 at 0x1465c010f390>,
 'BNCI2015-001': <moabb.datasets.bnci.BNCI2015_001 at 0x1465c010f410>,
 'BNCI2015-004': <moabb.datasets.bnci.BNCI2015_004 at 0x1465c0282d10>,
 'Cho2017': <moabb.datasets.gigadb.Cho2017 at 0x1465c0281d50>,
 'FakeDataset-imagery-10-2--60-60--120-120--fake1-fake2-fake3--c3-cz-c4': <moabb.datasets.fake.FakeDataset at 0x1465c0990e50>,
 'GrosseWentrup2009': <moabb.datasets.mpi_mi.GrosseWentrup2009 at 0x1465c010ed10>,
 'Lee2019-MI': <moabb.datasets.Lee2019.Lee2019_MI at 0x14661cfa9d90>,
 'Liu2024': <moabb.datasets.liu2024.Liu2024 at 0x1465c00f6250>,
 'Ofner2017': <moabb.datasets.upper_limb.Ofner2017 at 0x1465c010ee90>,
 'PhysionetMotorImagery': <moabb.datasets.physionet_mi.PhysionetMI at 0x1465c04b6550>,
 'S

In [2]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub

env: CUPY_ACCELERATORS=cutensor,cub


In [3]:
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: cupy


In [4]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from hoda.classification import ZScore, BTTDACV
from sklearn.preprocessing import StandardScaler
from mne.decoding import Scaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer


pipelines=dict()

cv=StratifiedKFold(random_state=42, shuffle=True)

hoda_params = dict(
    max_iter=256,
    tol=1e-6,
    shrinkage='lw',
    solver='lanczos',
    taper=False,
    forward=False,
    obj='tr',
    toeplitz=None,
    extra_train_info=False,
    refit_shrinkage=False,
    verbose=False,
    theta=None,
    rank=None
)

bttda_params=dict(
    hoda_params=hoda_params,
    extra_train_info=False,
    verbose=True,
)

clf = make_pipeline(
    FunctionTransformer(tl.base.unfold, kw_args=dict(mode=0)),
    FunctionTransformer(tl.to_numpy),
    StandardScaler(),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

pipelines['HODA'] = Pipeline([
    ('envelope', FunctionTransformer(envelope, kw_args=dict(sfreq=sfreq))),
    ('tl', FunctionTransformer(tl.tensor)),
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=1,
        thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        **bttda_params)
    ),
    ('clf', clf)
])

pipelines['BTTDA'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=2,
        thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        **bttda_params)
    ),
    ('clf', clf)
])
"""
pipelines['PARAFACDA'] = Pipeline([
    ('zscore1', ZScore()),
    ('bttda',BTTDACV(
        max_n_blocks=16,
        thetas=[0],
        **bttda_params)
    ),
    ('clf', clf)
], memory='.cache')
"""

NameError: name 'envelope' is not defined

In [ ]:
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from hoda.tensorize import stf_tensor

results = []
for dataset in datasets:
    print(dataset.code)
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=[dataset],
        overwrite=False,
        random_state=42,
        n_jobs=1,
        suffix=f'bttda_{dataset.code}',
    )
    results.append(evaluation.process(pipelines))
results = pd.concat(results)

In [ ]:
results

In [ ]:
results.to_csv('moabb_results_mi.csv')

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean') 

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('std') 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()



sns.catplot(data=results , col='session',x='subject', y='score',hue='pipeline', col_wrap=3,kind='bar')


In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)
plt.show()

In [ ]:
import moabb.analysis.plotting as moabb_plt
fig = moabb_plt.paired_plot(results, "PARAFACDA_10", "HODA")
plt.show()